# CURE-Rec — final required experiments

Run all cells. This notebook executes the remaining code-backed experiments and writes auditable manifests. It does not fabricate real intervention evidence. The divergent-selector experiment uses disclosed controlled regimes where masks differ; the real-intervention validation and integrated 8/10 policy study are reported as blocked unless their required data/operators are available.


In [ ]:
from pathlib import Path
import sys,json
import pandas as pd

C=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in C if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.observability import RunLogger
from cure_rec.regimes import run_regime_suite

CONFIG=ROOT/'configs'/'curesim_quickstart.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'/'final_required'
RESULTS.mkdir(parents=True,exist_ok=True)
RUN_ALL=True
RUN_DIVERGENT_SELECTOR=True
RUN_REAL_INTERVENTION=False
RUN_INTEGRATED_SCALING=False
print('Root:',ROOT)

## 1. Divergent-selector controlled benchmark

This runs the disclosed oracle regimes, records estimated/oracle masks and regret, and explicitly identifies regimes in which a simple selector cannot be assumed equivalent to direct robust selection. It is not presented as real-data evidence.

In [ ]:
if RUN_ALL and RUN_DIVERGENT_SELECTOR:
    settings=load_settings(CONFIG); logger=RunLogger(settings)
    try:
        result=run_regime_suite(settings,logger); logger.close(status='completed')
    except Exception:
        logger.close(status='failed'); raise
    summary=result.summary.copy()
    out=RESULTS/'divergent_selector_regimes.csv'; summary.to_csv(out,index=False)
    manifest={'scope':'controlled oracle regimes; selector divergence diagnostic, not real intervention evidence','source_run':str(result.run_dir),'regimes':summary['regime'].tolist(),'output':str(out)}
    (RESULTS/'divergent_selector_manifest.json').write_text(json.dumps(manifest,indent=2))
    display(summary)
else: print('Divergent selector study skipped.')

## 2. Real/semi-real intervention validation gate

CURE-Rec policy selection cannot be validated on MovieLens ratings alone. This cell only runs if an audited intervention log or declared replay/world-model input is present.

In [ ]:
if RUN_REAL_INTERVENTION:
    raise NotImplementedError('Provide audited logged slates, propensities, intervention assignment and outcomes before running this claim.')
else:
    print('Real/semi-real intervention validation blocked: no audited intervention log/world model is present.')

## 3. Integrated 8/10-player CURE scaling gate

The current 8/10 result is arithmetic attribution scaling. This cell prevents accidentally presenting it as integrated policy scaling until distinct operators are implemented in interventions.py, game.py and the simulator.

In [ ]:
required={'session_length_cap','freshness_quota','provider_cooldown','category_coverage_quota'}
implemented=set()
if RUN_INTEGRATED_SCALING and not required.issubset(implemented):
    raise NotImplementedError(f'Missing integrated operators: {sorted(required-implemented)}')
else:
    print('Integrated 8/10 policy scaling blocked until distinct operators are implemented; arithmetic benchmark remains separately labelled.')

## 4. Completion manifest


In [ ]:
manifest={'run_all':RUN_ALL,'divergent_selector':'executed_controlled_regimes' if RUN_DIVERGENT_SELECTOR else 'skipped','real_intervention':'blocked_no_audited_log','integrated_scaling':'blocked_missing_distinct_operators','yaml_changed':False,'claim_discipline':'blocked actions are not converted into claims'}
(RESULTS/'final_required_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))